# AI Warden — เก็บข้อมูลเทรนโมเดลของเราเอง

notebook นี้รวบรวมข้อมูลสำหรับ fine-tune โมเดลโค้ดตัวเล็ก (Qwen2.5-Coder-1.5B) ที่จะรันใน
`aider-local` ของ AI Warden แทนการจ่ายเงินให้ API ภายนอก วันนี้ทำแค่ **เก็บข้อมูล** — การเทรนอยู่ใน notebook ถัดไป

แหล่งข้อมูล 4 แหล่ง (เปิด/ปิดได้ในขั้นที่ 2):

| แหล่ง | ได้อะไร | ส่วนตัว? |
|---|---|---|
| git repo ของคุณ | commit message → diff ("ทำการเปลี่ยนแปลงนี้") | ใช่ |
| ไฟล์ของโปรเจกต์ | หัวข้อในเอกสาร → คำอธิบาย, docstring/comment → ฟังก์ชัน | แล้วแต่ repo |
| export แชท AI | บทสนทนาเรื่องโค้ดจาก Claude / ChatGPT | **ใช่** |
| Hugging Face | dataset code-instruct สาธารณะ (ตรวจ licence จาก card ตอนรัน) | ไม่ |

**กฎความเป็นส่วนตัว:** repo AI-Warden เป็น public — ข้อมูลที่ได้ต้องอยู่ใน `notebook/data/` (git ignore ไว้แล้ว)
หรือใน Google Drive ของคุณ **ห้าม commit** ทุก record ผ่านการลบ secret (API key, private key, token, email) ก่อนเขียน

## 1. Setup

In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mntoyg/AI-Warden"

if IN_COLAB:
    if not pathlib.Path("/content/AI-Warden").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/AI-Warden"], check=True)
    os.chdir("/content/AI-Warden/notebook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "huggingface_hub"], check=True)
elif pathlib.Path("collect").is_dir():
    pass                      # already in notebook/
elif pathlib.Path("notebook/collect").is_dir():
    os.chdir("notebook")      # opened from the repo root
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

## 2. ตั้งค่า

- `OWN_REPOS`: URL ของ repo คุณ (private ก็ได้ — ขั้นที่ 3 จะถาม token แบบไม่แสดงบนจอ และลบ token ออกจาก clone ทันที)
- `LOCAL_PATHS`: repo/โฟลเดอร์ที่อยู่ในเครื่องหรือใน Drive แล้ว
- `CHAT_EXPORTS`: path ของ `conversations.json` จาก export ของ Claude หรือ ChatGPT
- `HF_DATASETS`: dataset สาธารณะ + จำนวนสูงสุด + ภาษาที่เอา
- `OUT_DIR`: ที่เก็บผลลัพธ์ — บน Colab แนะนำให้เก็บใน Drive (`USE_DRIVE = True`)

In [ ]:
OWN_REPOS = [
    "https://github.com/mntoyg/AI-Warden",
    # "https://github.com/mntoyg/Forge",          # private: needs a token in step 3
]
LOCAL_PATHS = []                                  # e.g. ["/content/drive/MyDrive/code/my-project"]
CHAT_EXPORTS = []                                 # e.g. ["/content/drive/MyDrive/exports/claude/conversations.json"]
CHATS_CODE_ONLY = True                            # keep only conversations that contain code
HF_DATASETS = [
    {"id": "ise-uiuc/Magicoder-OSS-Instruct-75K", "limit": 3000, "langs": {"python", "shell"}},
    # {"id": "bigcode/self-oss-instruct-sc2-exec-filter-50k", "limit": 2000, "langs": None},
]
USE_DRIVE = True
VAL_RATIO = 0.05

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/warden-model-lab/data"
else:
    OUT_DIR = "data"                              # notebook/data/ - ignored by git
print("output:", OUT_DIR)

## 3. Clone repo ของคุณ (token ไม่ถูกเก็บ)

In [ ]:
from getpass import getpass

CLONE_DIR = pathlib.Path("/content/repos" if IN_COLAB else "data/_repos")
CLONE_DIR.mkdir(parents=True, exist_ok=True)
token = None
repo_paths = list(LOCAL_PATHS)
for url in OWN_REPOS:
    name = url.rstrip("/").rsplit("/", 1)[-1].removesuffix(".git")
    dest = CLONE_DIR / name
    if not dest.exists():
        r = subprocess.run(["git", "clone", "-q", url, str(dest)], capture_output=True, text=True)
        if r.returncode != 0:            # private repo: ask once, never print or store it
            token = token or getpass("GitHub token (read-only is enough): ")
            authed = url.replace("https://", f"https://x-access-token:{token}@")
            subprocess.run(["git", "clone", "-q", authed, str(dest)], check=True, capture_output=True)
            subprocess.run(["git", "-C", str(dest), "remote", "set-url", "origin", url], check=True)
    repo_paths.append(str(dest))
token = None
print("\n".join(repo_paths))

## 4. ตรวจ dataset สาธารณะก่อนเก็บ (licence + คอลัมน์)

In [ ]:
from collect import from_hf

for spec in HF_DATASETS:
    lic = from_hf.card_license(spec["id"])
    ok = lic in from_hf.ALLOWED_LICENSES
    print(f"{spec['id']}: licence={lic!r} -> {'OK' if ok else 'REFUSED'}")
    if ok:
        from datasets import load_dataset
        first = next(iter(load_dataset(spec["id"], split="train", streaming=True)))
        print("   columns:", list(first), "| preset:", from_hf.PRESETS.get(spec["id"]))

## 5. เก็บข้อมูลทั้งหมด → train.jsonl / val.jsonl / manifest.json

In [ ]:
import json
from collect import build, from_chats, from_files, from_git

sources = {}
for p in repo_paths:
    sources[f"git:{pathlib.Path(p).name}"] = from_git.collect(p)
    sources[f"files:{pathlib.Path(p).name}"] = from_files.collect(p)
for p in CHAT_EXPORTS:
    sources[f"chats:{pathlib.Path(p).parent.name}"] = from_chats.collect(p, code_only=CHATS_CODE_ONLY)
for spec in HF_DATASETS:
    sources[f"hf:{spec['id']}"] = from_hf.collect(spec["id"], limit=spec["limit"], langs=spec["langs"])

manifest = build.build(sources, OUT_DIR, val_ratio=VAL_RATIO)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

## 6. ดูตัวอย่าง และตรวจว่าไม่มี secret หลุด

In [ ]:
import random, re
from collect import common

rows = list(common.read_jsonl(f"{OUT_DIR}/train.jsonl")) + list(common.read_jsonl(f"{OUT_DIR}/val.jsonl"))
by_source = {}
for r in rows:
    by_source.setdefault(r["source"], []).append(r)
for src, recs in sorted(by_source.items()):
    r = random.choice(recs)
    print(f"=== {src} ({len(recs)} records)")
    for m in r["messages"][1:]:
        print(f"[{m['role']}] {m['content'][:300]}")
    print()

# A second pass over what was written: any pattern still matching is a bug in redact().
leaks = [(r["source"], kind) for r in rows for m in r["messages"]
         for kind in common.find_leaks(m["content"])]
print("secret-shaped strings left:", len(leaks), leaks[:5])
assert not leaks, "redaction missed something - do not train on this file"

## 7. ต่อไป

- ไฟล์ใน `OUT_DIR` คือ input ของ notebook เทรน (LoRA บน Qwen2.5-Coder-1.5B → merge → GGUF → `aider-local`)
- เก็บ `manifest.json` ไว้คู่กับข้อมูลเสมอ: มีจำนวน record ต่อแหล่ง และ sha256 ของแต่ละไฟล์
- **ห้าม** `git add` ไฟล์ใน `data/` — ถ้า `git status` เห็นมัน แปลว่า `.gitignore` ไม่ทำงาน ให้หยุดก่อน